In [1]:
# Interfaz
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import numpy as np
import scipy.signal as sig
import NNMF_GIF2 as CONMF
import NNMF_GIF as NMF
import Funciones as Funciones
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial
%config InlineBackend.figure_format = 'svg'
from MetricasRendimiento import MetricasRendimiento
import ray
from matplotlib.backends.backend_pdf import PdfPages
import textwrap
import pandas as pd

#========================================================================================================================================
# Variables generales
#========================================================================================================================================
cmap = Funciones.parula_map
color_ACC = 'blue'
color_EGG = 'orange'
color_dEGG = 'g'
color_OVV = 'magenta'
color_speech = 'black'
legend_loc = 'upper right'
color1 = 'red'
color2 = 'blue'
color3 = 'k'
color_notch = 'green'
color_lpc  = 'orange'
color_flujo = 'k'
line_width = 1.5

@ray.remote
def procesar_archivo(nombre, carpeta_base, parametros):

    ruta_archivo = Path(carpeta_base) / nombre
    df = pd.read_csv(ruta_archivo, sep=',')
    
    fs_r = 8000
    tiempos = df['t_ms'].values

    # ==================================================================

    señal_MBK = df['MBK'].values
    señal_MBK = Funciones.Detrend(tiempos, señal_MBK)
    señal_MBK = señal_MBK - np.mean(señal_MBK)

    señal_ACC = -df['ACC'].values
    señal_ACC = señal_ACC - np.mean(señal_ACC)
    señal_ACC = Funciones.Detrend(tiempos, señal_ACC)
    señal_ACC = señal_ACC - np.mean(señal_ACC)

    señal_EGG = df['MBK'].values
    señal_EGG = Funciones.Detrend(tiempos, señal_EGG)
    señal_dEGG = np.gradient(señal_EGG)

    flujo_OVV = df['flujo_OVV'].values
    flujo_OVV = Funciones.Detrend(tiempos, flujo_OVV)
    dflujo_OVV = np.gradient(flujo_OVV)

    # ==================================================================
    # CONMF

    resultado_CONMF = CONMF.NNMF_GIF(speech=señal_MBK,acelerometro=señal_ACC,dEGG=señal_dEGG,tiempos=tiempos,fs=fs_r,**parametros["CONMF"])
    
    resultado_CONMF.flujo1_speech = Funciones.Detrend(resultado_CONMF.tiempos_gve_speech,resultado_CONMF.flujo1_speech)
    resultado_CONMF.flujo2_speech = Funciones.Detrend(resultado_CONMF.tiempos_gve_speech,resultado_CONMF.flujo2_speech)

    resultado_CONMF.flujo1_acc = Funciones.Detrend(resultado_CONMF.tiempos_gve_acc,resultado_CONMF.flujo1_acc)
    resultado_CONMF.flujo2_acc = Funciones.Detrend(resultado_CONMF.tiempos_gve_acc,resultado_CONMF.flujo2_acc)
    # ==================================================================
    # NMF SPEECH

    resultado_NMF_speech = NMF.NNMF_GIF(señal=señal_MBK,dEGG=señal_dEGG,tiempos=tiempos,fs=fs_r,**parametros["NMF_speech"])

    resultado_NMF_speech.flujo1 = Funciones.Detrend(resultado_NMF_speech.tiempos_gve,resultado_NMF_speech.flujo1)
    resultado_NMF_speech.flujo2 = Funciones.Detrend(resultado_NMF_speech.tiempos_gve,resultado_NMF_speech.flujo2)

    # ==================================================================
    # NMF ACC

    resultado_NMF_acc = NMF.NNMF_GIF(señal=señal_ACC,dEGG=señal_dEGG,tiempos=tiempos,fs=fs_r,**parametros["NMF_acc"])

    resultado_NMF_acc.flujo1 = Funciones.Detrend(resultado_NMF_acc.tiempos_gve,resultado_NMF_acc.flujo1)
    resultado_NMF_acc.flujo2 = Funciones.Detrend(resultado_NMF_acc.tiempos_gve,resultado_NMF_acc.flujo2)
    # ==================================================================
    # METRICAS

    MR_NMF_speech_1 = MetricasRendimiento(dflujo_OVV,resultado_NMF_speech.dflujo1,tiempos)
    MR_NMF_speech_2 = MetricasRendimiento(dflujo_OVV,resultado_NMF_speech.dflujo2,tiempos)
    
    MR_NMF_acc_1 = MetricasRendimiento(dflujo_OVV,resultado_NMF_acc.dflujo1,tiempos)
    MR_NMF_acc_2 = MetricasRendimiento(dflujo_OVV,resultado_NMF_acc.dflujo2,tiempos)

    MR_CONMF_speech_1 = MetricasRendimiento(dflujo_OVV,resultado_CONMF.dflujo1_speech,tiempos)
    MR_CONMF_speech_2 = MetricasRendimiento(dflujo_OVV,resultado_CONMF.dflujo2_speech,tiempos)

    MR_CONMF_acc_1 = MetricasRendimiento(dflujo_OVV,resultado_CONMF.dflujo1_acc,tiempos)

    MR_CONMF_acc_2 = MetricasRendimiento(dflujo_OVV,resultado_CONMF.dflujo2_acc,tiempos)
    return {
        "NMF_speech_1": MR_NMF_speech_1.Error,
        "NMF_speech_2": MR_NMF_speech_2.Error,
        "NMF_acc_1": MR_NMF_acc_1.Error,
        "NMF_acc_2": MR_NMF_acc_2.Error,
        "CONMF_speech_1": MR_CONMF_speech_1.Error,
        "CONMF_speech_2": MR_CONMF_speech_2.Error,
        "CONMF_acc_1": MR_CONMF_acc_1.Error,
        "CONMF_acc_2": MR_CONMF_acc_2.Error
    }

#========================================================================================================================================
def ejecutar(b):

    global error_NMF_speech_1
    global error_NMF_speech_2
    global error_NMF_acc_1
    global error_NMF_acc_2
    global error_CONMF_speech_1
    global error_CONMF_speech_2
    global error_CONMF_acc_1
    global error_CONMF_acc_2

    error_NMF_speech_1 = []
    error_NMF_speech_2 = []

    error_NMF_acc_1 = []
    error_NMF_acc_2 = []

    error_CONMF_speech_1 = []
    error_CONMF_speech_2 = []

    error_CONMF_acc_1 = []
    error_CONMF_acc_2 = []

    # ==============================================================
    # Arranco el Ray

    ray.shutdown()
    ray.init(ignore_reinit_error=True)

    # ==============================================================

    lista_señales = Path(fc.selected)
    carpeta_base = lista_señales.parent

    with open(lista_señales, "r", encoding="utf-8") as f:
        nombres_archivos = [linea.strip() for linea in f if linea.strip() ]

    total_archivos = len(nombres_archivos)
    barra_progreso.max = total_archivos
    barra_progreso.value = 0

    label_progreso.value = f'0/{total_archivos}'

    # ==============================================================
    parametros = {
        "CONMF": {
            "norma": conmf_norma.value,
            "pre_iteraciones": conmf_pre_iteraciones.value,
            "iteraciones": conmf_iteraciones.value,
            "p_filtro_speech": conmf_p_filtro_speech.value,
            "alfa_speech": conmf_alfa_speech.value,
            "p_filtro_acc": conmf_p_filtro_acc.value,
            "alfa_acc": conmf_alfa_acc.value,
            "beta_speech": conmf_beta_speech.value,
            "beta_acc": conmf_beta_acc.value,
            "pre_enfasis_speech": conmf_pre_enfasis_speech.value,
            "pre_enfasis_acc": conmf_pre_enfasis_acc.value,
            "duracion": conmf_duracion_ventana.value,
            "guardar_proceso": conmf_guardar_proceso.value,
            "normalizar_espectrogramas": conmf_normalizar_espectrogramas.value,
            "ventana": conmf_ventana.value
        },
        "NMF_speech": {
            "norma": nmf_norma.value,
            "pre_iteraciones": nmf_pre_iteraciones.value,
            "iteraciones": nmf_iteraciones.value,
            "p_filtro": nmf_p_filtro_speech.value,
            "alfa": nmf_alfa_speech.value,
            "beta": nmf_beta_speech.value,
            "pre_enfasis": nmf_pre_enfasis_speech.value,
            "duracion": nmf_duracion_ventana.value,
            "guardar_proceso": nmf_guardar_proceso.value,
            "ventana": nmf_ventana.value
        },
        "NMF_acc": {

            "norma": nmf_norma.value,
            "pre_iteraciones": nmf_pre_iteraciones.value,
            "iteraciones": nmf_iteraciones.value,
            "p_filtro": nmf_p_filtro_acc.value,
            "alfa": nmf_alfa_acc.value,
            "beta": nmf_beta_acc.value,
            "pre_enfasis": nmf_pre_enfasis_acc.value,
            "duracion": nmf_duracion_ventana.value,
            "guardar_proceso": nmf_guardar_proceso.value,
            "ventana": nmf_ventana.value
        }
    }

    # ==============================================================
    # Ejecuto las tareas
    futures = [procesar_archivo.remote(nombre,str(carpeta_base),parametros) for nombre in nombres_archivos ]

    # ==============================================================
    # Obtengo los resultados
    restantes = futures
    procesados = 0
    while restantes:
        terminados, restantes = ray.wait(restantes)
        resultado = ray.get(terminados[0])

        error_CONMF_speech_1.append(resultado["CONMF_speech_1"])
        error_CONMF_speech_2.append(resultado["CONMF_speech_2"])

        error_CONMF_acc_1.append(resultado["CONMF_acc_1"])
        error_CONMF_acc_2.append(resultado["CONMF_acc_2"])

        error_NMF_speech_1.append(resultado["NMF_speech_1"])
        error_NMF_speech_2.append(resultado["NMF_speech_2"])

        error_NMF_acc_1.append(resultado["NMF_acc_1"])
        error_NMF_acc_2.append(resultado["NMF_acc_2"])

        procesados += 1

        barra_progreso.value = procesados

        label_progreso.value = (f'{procesados}/{total_archivos}')

    with tab_boxplots:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12,6))
        facecolor1 = "#ff7e6a"
        facecolor2 = "#6c7dff"

        datos = [
            np.array(error_NMF_speech_1)[~np.isnan(error_NMF_speech_1)],
            np.array(error_NMF_speech_2)[~np.isnan(error_NMF_speech_2)],
            np.array(error_NMF_acc_1)[~np.isnan(error_NMF_acc_1)],
            np.array(error_NMF_acc_2)[~np.isnan(error_NMF_acc_2)],
            np.array(error_CONMF_speech_1)[~np.isnan(error_CONMF_speech_1)],
            np.array(error_CONMF_speech_2)[~np.isnan(error_CONMF_speech_2)],
            np.array(error_CONMF_acc_1)[~np.isnan(error_CONMF_acc_1)],
            np.array(error_CONMF_acc_2)[~np.isnan(error_CONMF_acc_2)]
        ]

        labels = [
            'NMF - S1',
            'NMF - S2',
            'NMF - A1',
            'NMF - A2',
            'CONMF - S1',
            'CONMF - S2',
            'CONMF - A1',
            'CONMF - A2'
        ]

        bp = ax.boxplot(
            datos,
            labels=labels,
            showmeans=False,
            showfliers=False,
            patch_artist=True
        )
        for box, label in zip(bp['boxes'], labels):

            if '1' in label:
                box.set_facecolor(facecolor1)

            elif '2' in label:
                box.set_facecolor(facecolor2)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)

        ax.set_ylabel(r"Errorr $E_{vg}$")
        ax.grid(True, axis='y', linestyle='--', alpha=0.5)
        plt.show()
    # ==========================================================
    # Genero el pdf

    nombre_pdf = "Boxplots_"+lista_señales.name.removesuffix(".txt")+".pdf"

    with PdfPages(nombre_pdf) as pdf:

        # ------------------------------------------------------
        # Página 1 - La figura

        pdf.savefig(fig, bbox_inches='tight')

        # ------------------------------------------------------
        # Página 2 - La información

        fig_texto = plt.figure(figsize=(8.27, 11.69))  # A4

        texto = ""

        texto += "Archivo:\n"
        texto += "============================\n\n"
        texto += f"{lista_señales.name.removesuffix(".txt")}\n\n"
        texto += "\n\n"
        texto += "Cantidad de ventanas analizadas:\n"
        texto += "============================\n\n"
        texto += f"{total_archivos}\n\n"
        texto += "\n\n"
        texto += "Parámetros\n"
        texto += "============================\n\n"

        for grupo, diccionario in parametros.items():

            texto += f"[{grupo}]\n\n"

            for clave, valor in diccionario.items():

                texto += f"{clave}: {valor}\n"

            texto += "\n"

        # wrap para que no se salga del ancho
        texto = "\n".join(
            textwrap.fill(linea, width=95)
            for linea in texto.split("\n")
        )

        fig_texto.text(
            0.03,
            0.97,
            texto,
            va='top',
            ha='left',
            fontsize=10,
            family='monospace'
        )

        plt.axis('off')

        pdf.savefig(fig_texto, bbox_inches='tight')

        plt.close(fig_texto)


#========================================================================================================================================
# Títulos de las secciones
#========================================================================================================================================
titulo_archivo = widgets.HTML(
    value="<b>Archivo</b>"
)
titulo_CONMF = widgets.HTML(
    value="""
    <div style='width:100%; font-size:36px; font-weight:bold; background-color:#8080ff; color:white; padding:10px;'>
        CO-NMF
    </div>
    """
)
titulo_NMF = widgets.HTML(
    value="""
    <div style='width:100%; font-size:36px; font-weight:bold; background-color:#ff7878; color:white; padding:10px;'>
        NMF
    </div>
    """
)
subtitulo_espectrograma = widgets.HTML(
    value="<b>Espectrograma</b>"
)
subtitulo_speech= widgets.HTML(
    value="<b>Speech</b>"
)
subtitulo_acc= widgets.HTML(
    value="<b>ACC</b>"
)
subtitulo_nmf= widgets.HTML(
    value="<b>NNMF</b>"
)
subtitulo_gif= widgets.HTML(
    value="<b>GIF</b>"
)
linea_divisoria = widgets.HTML(
    value="<hr style='border:1px solid #ccc; margin:10px 0;'>"
)
#========================================================================================================================================
# Archivo y botón
#========================================================================================================================================

fc = FileChooser('Señales VEC')
fc.layout = widgets.Layout(width='auto')

boton_ejecutar = widgets.Button(
    description="Ejecutar",
    button_style='success'
)

barra_progreso = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    step=1,
    description='Progreso:',
    bar_style='info',
    orientation='horizontal',
    layout=widgets.Layout(width='300px')
)

label_progreso = widgets.Label(value='0/0')

grid_archivo_boton = widgets.GridBox(
    children=[fc, boton_ejecutar, barra_progreso, label_progreso],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_gap='10px 20px',
        width='90%'
    )
)
#========================================================================================================================================
# Elementos de CO-NMF
#========================================================================================================================================
conmf_norma = widgets.Dropdown(
    options=['frobenius','itakura-saito'],
    value='frobenius',
    description='Norma:',
    layout=widgets.Layout(width='300px')
)
conmf_ventana = widgets.Dropdown(
    options=['Rectangular','Hanning','Blackman','Hamming'],
    value='Rectangular',
    description='Ventana:',
    layout=widgets.Layout(width='200px')
)
conmf_duracion_ventana = widgets.BoundedFloatText(
    value=5.5,
    min=1.0,
    max=20.0,
    step=0.5,
    description='Duración ventana (ms):',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
conmf_desplazamiento = widgets.BoundedIntText(
    value=1,
    min=1,
    max=10,
    step=1,
    description='hop:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px'),
)
conmf_pre_iteraciones = widgets.BoundedIntText(
    value=0,
    min=0,
    max=100,
    step=1,
    description='Pre-iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)
conmf_iteraciones = widgets.BoundedIntText(
    value=250,
    min=1,
    max=50000,
    step=1,
    description='Iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)
conmf_p_filtro_speech = widgets.BoundedIntText(
    value=12,
    min=2,
    max=20,
    step=1,
    description='Orden del filtro del tracto:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
conmf_p_filtro_acc = widgets.BoundedIntText(
    value=12,
    min=2,
    max=20,
    step=1,
    description='Orden del filtro del tracto:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
conmf_alfa_speech = widgets.BoundedFloatText(
    value=0.96,
    min=0.0,
    max=1.0,
    step=0.01,
    description='α:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px')
)
conmf_alfa_acc = widgets.BoundedFloatText(
    value=0.96,
    min=0.0,
    max=1.0,
    step=0.01,
    description='α:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px')
)
conmf_beta_speech = widgets.BoundedFloatText(
    value=0.55,
    min=0.1,
    max=2.0,
    step=0.05,
    description='β:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)
conmf_beta_acc = widgets.BoundedFloatText(
    value=0.55,
    min=0.1,
    max=2.0,
    step=0.05,
    description='β:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)
conmf_pre_enfasis_speech = widgets.Checkbox(
    value=True,
    description='Pre-énfasis'
)
conmf_pre_enfasis_acc = widgets.Checkbox(
    value=True,
    description='Pre-énfasis'
)
conmf_guardar_proceso = widgets.Checkbox(
    value=True,
    description='Guardar proceso'
)
conmf_normalizar_espectrogramas = widgets.Checkbox(
    value=True,
    description='Normalizar espectrogramas'
)
conmf_grid_speech = widgets.GridBox(
    children=[conmf_pre_enfasis_speech, conmf_beta_speech,conmf_p_filtro_speech,conmf_alfa_speech],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',  
        grid_gap='0px 5px',        
        width='60%'
    )
)
conmf_grid_acc = widgets.GridBox(
    children=[conmf_pre_enfasis_acc, conmf_beta_acc,conmf_p_filtro_acc,conmf_alfa_acc],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 5px',
        width='60%'
    )
)
conmf_grid_espectrograma = widgets.GridBox(
    children=[conmf_ventana, conmf_duracion_ventana, conmf_desplazamiento,conmf_normalizar_espectrogramas],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',  
        width='60%'
    )
)
conmf_grid_nmf = widgets.GridBox(
    children=[conmf_norma, conmf_iteraciones],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',   
        width='60%'
    )
)

#========================================================================================================================================
# Elementos de NMF
#========================================================================================================================================
nmf_norma = widgets.Dropdown(
    options=['frobenius','itakura-saito'],
    value='frobenius',
    description='Norma:',
    layout=widgets.Layout(width='300px')
)
nmf_ventana = widgets.Dropdown(
    options=['Rectangular','Hanning','Blackman','Hamming'],
    value='Rectangular',
    description='Ventana:',
    layout=widgets.Layout(width='200px')
)
nmf_duracion_ventana = widgets.BoundedFloatText(
    value=5.5,
    min=1.0,
    max=20.0,
    step=0.5,
    description='Duración ventana (ms):',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
nmf_desplazamiento = widgets.BoundedIntText(
    value=1,
    min=1,
    max=10,
    step=1,
    description='hop:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px'),
)
nmf_pre_iteraciones = widgets.BoundedIntText(
    value=0,
    min=0,
    max=100,
    step=1,
    description='Pre-iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)
nmf_iteraciones = widgets.BoundedIntText(
    value=250,
    min=1,
    max=50000,
    step=1,
    description='Iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)
nmf_p_filtro_speech = widgets.BoundedIntText(
    value=12,
    min=2,
    max=20,
    step=1,
    description='Orden del filtro del tracto:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
nmf_alfa_speech = widgets.BoundedFloatText(
    value=0.96,
    min=0.0,
    max=1.0,
    step=0.01,
    description='α:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px')
)
nmf_p_filtro_acc = widgets.BoundedIntText(
    value=12,
    min=2,
    max=20,
    step=1,
    description='Orden del filtro del tracto:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
nmf_alfa_acc = widgets.BoundedFloatText(
    value=0.96,
    min=0.0,
    max=1.0,
    step=0.01,
    description='α:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px')
)
nmf_beta_speech = widgets.BoundedFloatText(
    value=0.55,
    min=0.1,
    max=2.0,
    step=0.05,
    description='β:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)
nmf_beta_acc = widgets.BoundedFloatText(
    value=0.55,
    min=0.1,
    max=2.0,
    step=0.05,
    description='β:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)
nmf_pre_enfasis_speech = widgets.Checkbox(
    value=True,
    description='Pre-énfasis'
)
nmf_pre_enfasis_acc = widgets.Checkbox(
    value=True,
    description='Pre-énfasis'
)
nmf_guardar_proceso = widgets.Checkbox(
    value=True,
    description='Guardar proceso'
)
nmf_grid_speech = widgets.GridBox(
    children=[nmf_pre_enfasis_speech, nmf_beta_speech,nmf_p_filtro_speech,nmf_alfa_speech],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',  
        grid_gap='0px 5px',
        width='60%'
    )
)
nmf_grid_acc = widgets.GridBox(
    children=[nmf_pre_enfasis_acc, nmf_beta_acc,nmf_p_filtro_acc,nmf_alfa_acc],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 5px',
        width='60%'
    )
)
nmf_grid_espectrograma = widgets.GridBox(
    children=[nmf_ventana, nmf_duracion_ventana, nmf_desplazamiento],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',  
        width='60%'
    )
)
nmf_grid_nmf = widgets.GridBox(
    children=[nmf_norma, nmf_iteraciones],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',  
        width='60%'
    )
)

#========================================================================================================================================
# Ensamble de la estructura de la intefaz
#========================================================================================================================================
estructura_interfaz = widgets.VBox([
    titulo_archivo,
    grid_archivo_boton,
    linea_divisoria,
    titulo_CONMF,
    linea_divisoria,
    subtitulo_speech,
    conmf_grid_speech,
    linea_divisoria,
    subtitulo_acc,
    conmf_grid_acc,
    linea_divisoria,
    subtitulo_espectrograma,
    conmf_grid_espectrograma,
    linea_divisoria,
    subtitulo_nmf,
    conmf_grid_nmf,
    linea_divisoria,
    titulo_NMF,
    linea_divisoria,
    subtitulo_speech,
    nmf_grid_speech,
    linea_divisoria,
    subtitulo_acc,
    nmf_grid_acc,
    linea_divisoria,
    subtitulo_espectrograma,
    nmf_grid_espectrograma,
    linea_divisoria,
    subtitulo_nmf,
    nmf_grid_nmf,
    linea_divisoria
])
#========================================================================================================================================
# Configuración de las pestañas
#========================================================================================================================================
tab_boxplots = widgets.Output()


tab = widgets.Tab(children=[
    estructura_interfaz,
    tab_boxplots
])
tab.set_title(0,'Configuración')
tab.set_title(1,'Boxplots')

boton_ejecutar.on_click(ejecutar)
display(tab)